# IBM Stock Price Forecasting

This project explores IBM historical stock data and develops a model to
predict the next trading-day closing price using information available
at the current trading day's close.

- **Reported data source:** Yahoo Finance
- **Databricks table:** `workspace.default.ibm_stock_raw`
- **Provisional target:** `Close`
- **Forecast horizon:** One trading day

The original table will remain unchanged. Price-adjustment conventions
will be checked before modeling.

%md
## 1. Load and preview the dataset

Read the imported table into a Spark DataFrame and display the earliest
10 records in chronological order.

This initial inspection helps us understand the columns and confirm
that the data is readable.

In [0]:
df = spark.table("workspace.default.ibm_stock_raw")

display(df.orderBy("Date").limit(10))

## 2. Inspect column types

Check how Databricks interpreted each column.

We expect:
- `Date` to represent a date.
- Price, dividend, and split columns to contain numeric values.
- `Volume` to contain whole-number values.

If a column has an unsuitable type, we will convert it during preprocessing.

In [0]:
df.printSchema()

## 3. Verify the row count and date coverage

Compare the imported table with the original CSV.

| Check | Expected value |
|---|---|
| Number of records | 16,272 |
| Earliest date | 1962-01-02 |
| Latest date | 2026-08-27 |

These checks help confirm the import. They do not establish that all
expected trading sessions are present or that every price is accurate.

If `Date` is stored as text, these date bounds are provisional until
we validate and convert it.

In [0]:
from pyspark.sql import functions as F

display(
    df.agg(
        F.count("*").alias("row_count"),
        F.min("Date").alias("first_date"),
        F.max("Date").alias("last_date")
    )
)

## 4. Standardize column names and data types

Create a separate DataFrame for preprocessing while keeping the original
table unchanged.

We will:
- Convert column names to lowercase and replace spaces with underscores.
- Convert `date` to a date type.
- Convert prices, dividends, and split values to numeric types.
- Store volume as a whole number.

Values that cannot be converted will become null and will be investigated
in the next step.

In [0]:
from pyspark.sql import functions as F

clean_df = df.toDF(
    *[
        name.strip().lower().replace(" ", "_")
        for name in df.columns
    ]
)

decimal_columns = [
    "open", "high", "low", "close",
    "adj_close", "dividends", "stock_splits"
]

clean_df = clean_df.withColumn(
    "date", F.expr("try_cast(date AS DATE)")
)

for name in decimal_columns:
    clean_df = clean_df.withColumn(
        name, F.expr(f"try_cast(`{name}` AS DOUBLE)")
    )

clean_df = clean_df.withColumn(
    "volume", F.expr("try_cast(volume AS BIGINT)")
)

clean_df.printSchema()
display(clean_df.orderBy("date").limit(5))

## 5. Check missing and invalid numeric values

Count null values in every column. For decimal columns, also count
NaN and positive or negative infinity.

A missing value may have existed in the source or resulted from a failed
type conversion.

We will inspect affected records before deciding how to handle them.
Prices will not be automatically filled with zero or interpolated.

In [0]:
invalid_conditions = {}

for name in clean_df.columns:
    condition = F.col(name).isNull()

    if name in decimal_columns:
        condition = (
            condition
            | F.isnan(F.col(name))
            | (F.abs(F.col(name)) == F.lit(float("inf")))
        )

    invalid_conditions[name] = condition

quality_summary = clean_df.agg(
    F.count("*").alias("total_rows"),
    *[
        F.sum(condition.cast("int")).alias(f"{name}_invalid")
        for name, condition in invalid_conditions.items()
    ]
)

display(quality_summary)

# Show records with at least one missing or invalid value.
any_invalid = F.lit(False)

for condition in invalid_conditions.values():
    any_invalid = any_invalid | condition

invalid_value_rows = clean_df.filter(any_invalid)

display(invalid_value_rows.orderBy("date"))

## 6. Check duplicate records and dates

For this single-stock dataset, we expect one record per trading date.

We will distinguish:
- Exact duplicate rows, where every field is identical.
- Repeated dates, which may contain conflicting values.

Duplicates will be inspected before removal. Keeping an arbitrary row
could discard the correct observation.

In [0]:
total_rows = clean_df.count()
distinct_rows = clean_df.distinct().count()

print("Total rows:", total_rows)
print("Exact duplicate rows beyond the first copy:", total_rows - distinct_rows)

duplicate_dates = (
    clean_df
    .filter(F.col("date").isNotNull())
    .groupBy("date")
    .count()
    .filter(F.col("count") > 1)
)

display(duplicate_dates.orderBy("date"))

# Show all records belonging to repeated dates.
duplicate_records = clean_df.join(
    duplicate_dates.select("date"),
    on="date",
    how="inner"
)

display(duplicate_records.orderBy("date"))

## 7. Check price and volume consistency

For each trading day, we expect:
- Prices to be positive.
- The highest price to be at least the lowest price.
- Opening and closing prices to fall within the day's low–high range.
- Trading volume to be nonnegative.

Adjusted close is checked for positivity, but not against the day's
low–high range because it may use a different adjustment basis.

Flagged records will be investigated before any changes are made.
These checks do not establish whether the prices match the source.

In [0]:
price_columns = ["open", "high", "low", "close", "adj_close"]

nonpositive_price = F.lit(False)

for name in price_columns:
    nonpositive_price = nonpositive_price | (F.col(name) <= 0)

price_checks = {
    "nonpositive_price": nonpositive_price,
    "high_below_low": F.col("high") < F.col("low"),
    "open_outside_range": (
        (F.col("open") < F.col("low"))
        | (F.col("open") > F.col("high"))
    ),
    "close_outside_range": (
        (F.col("close") < F.col("low"))
        | (F.col("close") > F.col("high"))
    ),
    "negative_volume": F.col("volume") < 0
}

display(
    clean_df.agg(
        *[
            F.sum(condition.cast("int")).alias(name)
            for name, condition in price_checks.items()
        ]
    )
)

any_price_issue = F.lit(False)

for condition in price_checks.values():
    any_price_issue = any_price_issue | condition

display(
    clean_df
    .filter(any_price_issue)
    .orderBy("date")
)

## 8. Inspect the trading-date sequence

Stock observations follow a trading calendar rather than a continuous
daily calendar. Weekends and exchange holidays usually have no records.

We will inspect:
- Observations dated on weekends.
- Calendar-day gaps between consecutive observations.
- Gaps longer than three calendar days.

A long gap is a flag for investigation, not proof of missing data.
Confirming missing trading sessions requires comparison with the
appropriate exchange calendar.

We will not insert or fill prices for non-trading days.

In [0]:
from pyspark.sql.window import Window

# In Spark, dayofweek returns 1 for Sunday and 7 for Saturday.
weekend_rows = clean_df.filter(
    F.dayofweek("date").isin(1, 7)
)

print("Weekend observations:", weekend_rows.count())
display(weekend_rows.orderBy("date"))

date_window = Window.orderBy("date")

date_gaps = (
    clean_df
    .select("date")
    .withColumn(
        "previous_date",
        F.lag("date").over(date_window)
    )
    .withColumn(
        "gap_calendar_days",
        F.datediff("date", "previous_date")
    )
)

# Summarize how frequently each gap length occurs.
display(
    date_gaps
    .filter(F.col("previous_date").isNotNull())
    .groupBy("gap_calendar_days")
    .count()
    .orderBy("gap_calendar_days")
)

# Inspect longer gaps without removing any observations.
display(
    date_gaps
    .filter(F.col("gap_calendar_days") > 3)
    .orderBy("date")
)

## 9. Define the analysis periods

We will keep observations in chronological order:

- Training: dates before 1 January 2023.
- Validation: 1 January 2023 through 31 December 2024.
- Test: 1 January 2025 onward.

The training period will support exploratory analysis and initial model
development. Validation will help compare modeling choices. The test
period will be reserved for the final evaluation.

These boundaries apply to observation dates here. When we create
next-day prediction targets, we will assign examples to periods using
their target dates.

In [0]:
train_prices = clean_df.filter(
    F.col("date") < F.lit("2023-01-01").cast("date")
)

validation_prices = clean_df.filter(
    (F.col("date") >= F.lit("2023-01-01").cast("date"))
    & (F.col("date") < F.lit("2025-01-01").cast("date"))
)

test_prices = clean_df.filter(
    F.col("date") >= F.lit("2025-01-01").cast("date")
)

def period_summary(data, period_name):
    return data.agg(
        F.count("*").alias("rows"),
        F.min("date").alias("first_date"),
        F.max("date").alias("last_date")
    ).withColumn("period", F.lit(period_name))

summary = (
    period_summary(train_prices, "1. Training")
    .unionByName(period_summary(validation_prices, "2. Validation"))
    .unionByName(period_summary(test_prices, "3. Test"))
)

display(
    summary.select("period", "rows", "first_date", "last_date")
           .orderBy("period")
)

## 10. Plot the training-period closing price

We will examine two views:
1. The complete training history for long-term context.
2. The last five calendar years of training data for a closer view.

Look for:
- Changes in the price level.
- Sustained upward or downward movements.
- Abrupt jumps or drops.
- Whether larger price levels coincide with larger absolute fluctuations.
- Any apparently repeating patterns.

A price chart alone cannot establish stationarity, seasonality, or
forecastability. These observations will guide further analysis.

`Close` remains our provisional target because the source's exact
price-adjustment settings are unconfirmed.

In [0]:
import pandas as pd
import matplotlib.pyplot as plt

# This single-stock dataset is small enough to collect for plotting.
train_pd = (
    train_prices
    .select("date", "close")
    .orderBy("date")
    .toPandas()
)

train_pd["date"] = pd.to_datetime(train_pd["date"])
train_pd = train_pd.set_index("date")

recent_train = train_pd.loc["2018-01-01":]

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

axes[0].plot(
    train_pd.index,
    train_pd["close"],
    color="steelblue",
    linewidth=0.9
)
axes[0].set_title("IBM closing price — complete training history")
axes[0].set_ylabel("Close (source price units)")

axes[1].plot(
    recent_train.index,
    recent_train["close"],
    color="darkorange",
    linewidth=1
)
axes[1].set_title("IBM closing price — training observations, 2018–2022")
axes[1].set_ylabel("Close (source price units)")
axes[1].set_xlabel("Date")

for ax in axes:
    ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

## 11. Initial observations from the price plots

The complete training history shows substantial changes in the price
level and periods of sustained increases and decreases.

Later observations show larger absolute price movements, but the price
level is also higher. We need to examine proportional changes before
drawing conclusions about volatility.

The 2018–2022 view shows abrupt declines and recoveries. No clear,
regular seasonal pattern is visible at this scale.

These are preliminary visual observations. Stationarity and seasonality
will require further investigation.

## 12. Compare closing price with log closing price

A logarithmic transformation expresses price movements on a proportional
scale. Equal proportional changes produce equal changes in log price.

For example, an increase from 50 to 55 and an increase from 100 to 110
are both 10% increases.

We will compare the original and log-transformed training series.
Logging may reduce the dependence of fluctuation size on price level,
but it does not automatically remove trend or make a series stationary.

The original closing-price column will be preserved.

In [0]:
import numpy as np
import matplotlib.pyplot as plt

# Logarithms require positive, finite prices.
assert (
    np.isfinite(train_pd["close"]).all()
    and (train_pd["close"] > 0).all()
), "Check for missing, infinite, or nonpositive closing prices."

train_pd = train_pd.copy()
train_pd["log_close"] = np.log(train_pd["close"])

fig, axes = plt.subplots(
    2, 1,
    figsize=(14, 8),
    sharex=True
)

axes[0].plot(
    train_pd.index,
    train_pd["close"],
    color="steelblue",
    linewidth=0.9
)
axes[0].set_title("IBM closing price — training period")
axes[0].set_ylabel("Close (source price units)")

axes[1].plot(
    train_pd.index,
    train_pd["log_close"],
    color="seagreen",
    linewidth=0.9
)
axes[1].set_title("IBM log closing price — training period")
axes[1].set_ylabel("Natural log of close")
axes[1].set_xlabel("Date")

for ax in axes:
    ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

## 13. Interpretation of the log-price plot

The logarithmic transformation makes proportional price movements
during earlier periods more visible and compresses higher price levels.

However, substantial long-term changes in the level remain. The plot
does not establish that log price is stationary or that its variance
is constant.

Next, we will examine changes between successive log prices.

## 14. Calculate log returns and rolling volatility

The log return is the first difference of log closing price:

r(t) = log(P(t)) - log(P(t-1))

For small changes, 100 × log return approximately equals the percentage
price change. This approximation is less accurate for large changes.

We will plot:
- Log returns between consecutive recorded trading sessions.
- The trailing 20-observation standard deviation of log returns.

The rolling standard deviation describes recent daily volatility.
It is not annualized and uses only the current and preceding observations.

The first return is unavailable because there is no preceding price
within the training series. The first 19 valid returns do not yet have
a complete 20-return window.

These are changes in the supplied `Close` series, not a verified
dividend-inclusive total-return measure.

In [0]:
# Preserve the existing price and log-price columns.
train_pd["log_return"] = train_pd["log_close"].diff()

train_pd["rolling_volatility_20"] = (
    train_pd["log_return"]
    .rolling(window=20, min_periods=20)
    .std()
)

fig, axes = plt.subplots(
    2, 1,
    figsize=(14, 8),
    sharex=True
)

axes[0].plot(
    train_pd.index,
    100 * train_pd["log_return"],
    color="steelblue",
    linewidth=0.6
)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_title("IBM daily log returns — training period")
axes[0].set_ylabel("Log return × 100")

axes[1].plot(
    train_pd.index,
    100 * train_pd["rolling_volatility_20"],
    color="darkorange",
    linewidth=0.9
)
axes[1].set_title(
    "Trailing 20-observation standard deviation of daily log returns"
)
axes[1].set_ylabel("Standard deviation × 100")
axes[1].set_xlabel("Date")

for ax in axes:
    ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

## 15. Interpret returns and rolling volatility

Log returns fluctuate around a more stable-looking center than log
prices. However, their magnitude varies, with some periods showing
clusters of large movements.

The trailing 20-observation standard deviation confirms variation in
local volatility. Overlapping rolling windows and individual extreme
returns also contribute to the appearance of this curve.

These plots suggest that modeling the average return and modeling
volatility may require different approaches.

We have not yet established stationarity or independence.

In [0]:
%pip install statsmodels

In [0]:
import warnings
import pandas as pd
from statsmodels.tsa.stattools import adfuller, kpss

series_to_test = {
    "Log price": train_pd["log_close"],
    "Log return": train_pd["log_return"]
}

test_results = []

for series_name, values in series_to_test.items():
    values = values.dropna()

    # ADF: constant, with lag length selected using AIC.
    adf_result = adfuller(
    values,
    regression="c",
    autolag="AIC",
    result_object=False
    )

    test_results.append({
        "series": series_name,
        "test": "ADF",
        "statistic": adf_result[0],
        "reported_p_value": adf_result[1],
        "lags": adf_result[2],
        "decision_at_5pct": (
            "Reject unit root"
            if adf_result[1] < 0.05
            else "Do not reject unit root"
        ),
        "note": ""
    })

    # KPSS p-values are limited by its reference table.
    # Preserve any warnings so boundary values are not misread as exact.
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")

        kpss_result = kpss(
    values,
    regression="c",
    nlags="auto",
    result_object=False
    )

    test_results.append({
        "series": series_name,
        "test": "KPSS",
        "statistic": kpss_result[0],
        "reported_p_value": kpss_result[1],
        "lags": kpss_result[2],
        "decision_at_5pct": (
            "Reject level stationarity"
            if kpss_result[1] < 0.05
            else "Do not reject level stationarity"
        ),
        "note": " ".join(str(w.message) for w in caught)
    })

stationarity_results = pd.DataFrame(test_results)
display(stationarity_results)

## 17. Interpret the stationarity tests

For log price:
- ADF p = 0.6974: do not reject the unit-root hypothesis.
- KPSS p < 0.01: reject stationarity around a constant.

For log returns:
- ADF reports an extremely small p-value: reject the unit-root hypothesis.
- KPSS p > 0.10: do not reject stationarity around a constant.

Together, these results support using first-differenced log prices
(log returns) as a stationary modeling candidate.

We will not apply a second difference at this stage. These results
do not establish independence, normality, constant conditional
volatility, or stability across every historical subperiod.

For later modeling:
- ARIMA fitted to log price would provisionally use d = 1.
- ARIMA fitted directly to log returns would provisionally use d = 0,
  because the differencing has already been performed.

The lag counts reported by ADF and KPSS are test settings, not selected
ARIMA orders.

## 18. Examine autocorrelation in returns and squared returns

We will plot:
- ACF of log returns: correlation with earlier returns.
- PACF of log returns: correlation at each lag after accounting for
  intervening lags.
- ACF of squared log returns: dependence in the magnitude of movements.

Each lag represents one recorded trading observation.

The shaded regions are approximate 95% pointwise confidence bands.
Some spikes can occur by chance when many lags are inspected.
Changing volatility also warrants caution when interpreting these bands.

We will use the plots to suggest candidate models, not to choose a
final model solely from individual significant spikes.

In [0]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

returns = train_pd["log_return"].dropna()

fig, axes = plt.subplots(3, 1, figsize=(14, 11))

plot_acf(
    returns,
    lags=40,
    zero=False,
    alpha=0.05,
    ax=axes[0]
)
axes[0].set_title("ACF of IBM log returns — training period")

plot_pacf(
    returns,
    lags=40,
    zero=False,
    alpha=0.05,
    method="ywm",
    ax=axes[1]
)
axes[1].set_title("PACF of IBM log returns — training period")

plot_acf(
    returns.pow(2),
    lags=40,
    zero=False,
    alpha=0.05,
    ax=axes[2]
)
axes[2].set_title("ACF of squared IBM log returns — training period")

for ax in axes:
    ax.set_xlabel("Lag (trading observations)")
    ax.set_ylabel("Correlation")
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

## 19. Initial interpretation of autocorrelation

The ACF and PACF of log returns show small correlations and no clear
low-order pattern at the displayed scale.

The ACF of squared log returns shows persistent positive correlation.
This supports dependence in return magnitudes and possible volatility
clustering.

These findings suggest:
- A simple baseline should be a central forecasting benchmark.
- Complex mean models are not justified by these plots alone.
- Changing volatility may matter when constructing prediction intervals.

We will examine a closer view before proposing ARIMA candidates.
Statistical significance alone does not establish forecasting usefulness.

## 20. Inspect small correlations more closely

Replot the same diagnostics with narrower vertical limits.

This changes only the display, not the data or calculated correlations.
The confidence bands are approximate and pointwise; isolated crossings
can occur when many lags are inspected.

Look for the magnitude, persistence, and structure of the correlations,
rather than only whether a spike crosses a confidence band.

In [0]:
fig, axes = plt.subplots(3, 1, figsize=(14, 11))

plot_acf(
    returns,
    lags=40,
    zero=False,
    alpha=0.05,
    ax=axes[0]
)
axes[0].set_title("ACF of log returns — closer view")
axes[0].set_ylim(-0.10, 0.10)

plot_pacf(
    returns,
    lags=40,
    zero=False,
    alpha=0.05,
    method="ywm",
    ax=axes[1]
)
axes[1].set_title("PACF of log returns — closer view")
axes[1].set_ylim(-0.10, 0.10)

plot_acf(
    returns.pow(2),
    lags=40,
    zero=False,
    alpha=0.05,
    ax=axes[2]
)
axes[2].set_title("ACF of squared log returns — closer view")
axes[2].set_ylim(-0.05, 0.25)

for ax in axes:
    ax.set_xlabel("Lag (trading observations)")
    ax.set_ylabel("Correlation")
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

## 21. Conclusions from the autocorrelation plots

Log returns show small linear correlations, with several isolated
crossings of the approximate confidence bands. There is no clear
cutoff or repeating pattern that identifies a single model order.

Squared log returns show persistent positive autocorrelation,
supporting dependence in return magnitudes.

We will compare a small set of nonseasonal ARIMA models against a
naïve forecast. We will not choose large model orders or seasonal
differencing solely because of isolated significant lags.

Validation performance will determine whether additional complexity
improves forecasting.

## 22. Establish the naïve forecasting benchmark

For each validation trading day, predict its closing price using the
previous recorded trading day's close:

Predicted close(t) = Actual close(t-1)

This simulates a sequence of one-trading-day-ahead forecasts.
After each day ends, its actual closing price becomes available for
the next forecast.

The first validation forecast uses the final training close.
No final-test observations are used.

We will measure:
- MAE: mean absolute error, in the target's price units.
- RMSE: root mean squared error, which gives more weight to large errors.

These results establish the benchmark that candidate models must beat.

In [0]:
# Collect validation prices in chronological order.
validation_pd = (
    validation_prices
    .select("date", "close")
    .orderBy("date")
    .toPandas()
)

validation_pd["date"] = pd.to_datetime(validation_pd["date"])
validation_pd = validation_pd.set_index("date")

# Include the last training close to forecast the first validation day.
history_for_baseline = pd.concat([
    train_pd[["close"]].tail(1),
    validation_pd[["close"]]
]).sort_index()

assert history_for_baseline.index.is_unique, "Duplicate dates found."

history_for_baseline["prediction"] = (
    history_for_baseline["close"].shift(1)
)

# Evaluate only dates belonging to validation.
baseline_results = (
    history_for_baseline
    .loc[validation_pd.index]
    .rename(columns={"close": "actual"})
    .copy()
)

assert np.isfinite(
    baseline_results[["actual", "prediction"]].to_numpy(dtype=float)
).all(), "Missing or invalid actuals or predictions."

# Positive error means that the forecast was too low.
baseline_results["error"] = (
    baseline_results["actual"] - baseline_results["prediction"]
)

baseline_mae = baseline_results["error"].abs().mean()
baseline_rmse = np.sqrt(
    baseline_results["error"].pow(2).mean()
)

baseline_metrics = pd.DataFrame([{
    "model": "Naïve: previous close",
    "period": "Validation, 2023–2024",
    "forecasts": len(baseline_results),
    "MAE": baseline_mae,
    "RMSE": baseline_rmse
}])

display(baseline_metrics)

display(
    baseline_results.head(10)
    .reset_index()
)

## 23. Fit the first ARIMA candidate

We will fit ARIMA(1,1,0) to training-period log prices:

- p = 1: include one autoregressive lag after differencing.
- d = 1: difference log prices once, producing log returns.
- q = 0: include no moving-average terms.

We will use no drift term for this initial comparison.

Model parameters will be estimated using training observations only.
During validation, parameters will remain fixed, while the model state
will update after each new actual price becomes available.

We will use MAE as the primary comparison metric and RMSE as a
secondary measure of sensitivity to large errors.

In [0]:
from statsmodels.tsa.arima.model import ARIMA

# Arrays preserve observation order without imposing a calendar frequency.
training_log_prices = train_pd["log_close"].to_numpy(dtype=float)

arima_110_fit = ARIMA(
    training_log_prices,
    order=(1, 1, 0),
    trend="n"
).fit()

print(arima_110_fit.summary())

assert arima_110_fit.mle_retvals.get("converged", False), (
    "The model did not converge. Inspect the warnings before continuing."
)

## 24. Evaluate one-trading-day-ahead ARIMA forecasts

For each validation observation:

1. Forecast the next log closing price.
2. Convert the forecast back to price units.
3. Record the prediction.
4. Update the model state using the newly observed actual log price.

The actual price is supplied only after its forecast is recorded.
Parameters are not re-estimated during this initial experiment.

Exponentiating the log forecast gives the model-implied median price
under its Gaussian forecast assumption, not the mean price. This is
appropriate for our primary absolute-error comparison.

We will evaluate exactly the same dates as the naïve benchmark.
The final test period remains unused.

In [0]:
assert (
    np.isfinite(validation_pd["close"]).all()
    and (validation_pd["close"] > 0).all()
), "Validation prices must be positive and finite."

assert train_pd.index.max() < validation_pd.index.min()
assert validation_pd.index.is_monotonic_increasing

validation_log_prices = np.log(
    validation_pd["close"].to_numpy(dtype=float)
)

current_fit = arima_110_fit
price_predictions = []

for actual_log_price in validation_log_prices:
    # Forecast BEFORE providing this day's actual observation.
    predicted_log_price = float(
        np.asarray(current_fit.forecast(steps=1)).reshape(-1)[0]
    )

    price_predictions.append(np.exp(predicted_log_price))

    # Update the state, keeping the fitted parameters fixed.
    current_fit = current_fit.extend(
        np.array([actual_log_price])
    )

arima_results = pd.DataFrame(
    {
        "actual": validation_pd["close"].to_numpy(dtype=float),
        "prediction": price_predictions
    },
    index=validation_pd.index
)

assert arima_results.index.equals(baseline_results.index)
assert np.isfinite(arima_results.to_numpy()).all()

arima_results["error"] = (
    arima_results["actual"] - arima_results["prediction"]
)

arima_mae = arima_results["error"].abs().mean()
arima_rmse = np.sqrt(arima_results["error"].pow(2).mean())

comparison = pd.DataFrame([
    {
        "model": "Naïve: previous close",
        "forecasts": len(baseline_results),
        "MAE": baseline_mae,
        "RMSE": baseline_rmse
    },
    {
        "model": "ARIMA(1,1,0): log-price median",
        "forecasts": len(arima_results),
        "MAE": arima_mae,
        "RMSE": arima_rmse
    }
])

comparison["MAE_improvement_pct"] = (
    100 * (1 - comparison["MAE"] / baseline_mae)
)

comparison["RMSE_improvement_pct"] = (
    100 * (1 - comparison["RMSE"] / baseline_rmse)
)

display(comparison)

## 25. Interpret the first model comparison

ARIMA(1,1,0) produced slightly higher validation errors than the naïve
forecast:

- MAE was approximately 0.114% worse.
- RMSE was approximately 0.034% worse.

The differences are small, and we have not assessed their statistical
uncertainty. This experiment provides no evidence of an advantage from
adding one autoregressive term.

The naïve forecast remains preferred so far because it is simpler and
has lower validation errors.

We will now evaluate the remaining previously proposed candidates:
ARIMA(0,1,1) and ARIMA(1,1,1).

## 26. Compare the remaining ARIMA candidates

ARIMA(0,1,1) includes one moving-average term after differencing.
This term describes dependence on the previous model innovation
(forecast surprise), not a rolling average of prices.

ARIMA(1,1,1) includes both one autoregressive term and one
moving-average term.

For a consistent comparison, both models will:
- Use the same training-period log prices.
- Include no drift.
- Keep fitted parameters fixed during validation.
- Update their state after observing each actual value.
- Produce one-step-ahead median price forecasts.
- Be evaluated on the same 502 validation observations.

MAE remains the primary metric. The final test period remains unused.

In [0]:
# Retain fitted models and predictions for later diagnostics.
candidate_fits = {
    "ARIMA(1,1,0): log-price median": arima_110_fit
}

candidate_predictions = {
    "ARIMA(1,1,0): log-price median": arima_results.copy()
}

additional_metrics = []

for order in [(0, 1, 1), (1, 1, 1)]:
    name = f"ARIMA{order}: log-price median"
    print(f"Fitting {name}")

    fitted = ARIMA(
        training_log_prices,
        order=order,
        trend="n"
    ).fit()

    # Do not silently include unsuccessful fits in the ranking.
    if not fitted.mle_retvals.get("converged", False):
        print("Did not converge. Excluded pending investigation.")
        continue

    candidate_fits[name] = fitted
    current_fit = fitted
    predictions = []

    for actual_log_price in validation_log_prices:
        predicted_log_price = float(
            np.asarray(
                current_fit.forecast(steps=1)
            ).reshape(-1)[0]
        )

        predictions.append(np.exp(predicted_log_price))

        current_fit = current_fit.extend(
            np.array([actual_log_price])
        )

    results = pd.DataFrame(
        {
            "actual": validation_pd["close"].to_numpy(dtype=float),
            "prediction": predictions
        },
        index=validation_pd.index
    )

    assert results.index.equals(baseline_results.index)
    assert np.isfinite(results.to_numpy()).all()

    results["error"] = results["actual"] - results["prediction"]
    candidate_predictions[name] = results

    additional_metrics.append({
        "model": name,
        "forecasts": len(results),
        "MAE": results["error"].abs().mean(),
        "RMSE": np.sqrt(results["error"].pow(2).mean())
    })

all_comparisons = pd.concat(
    [
        comparison[["model", "forecasts", "MAE", "RMSE"]],
        pd.DataFrame(additional_metrics)
    ],
    ignore_index=True
)

all_comparisons["MAE_improvement_pct"] = (
    100 * (1 - all_comparisons["MAE"] / baseline_mae)
)

all_comparisons["RMSE_improvement_pct"] = (
    100 * (1 - all_comparisons["RMSE"] / baseline_rmse)
)

display(all_comparisons.sort_values("MAE").reset_index(drop=True))

## 27. Interpret the candidate-model comparison

The naïve forecast has the lowest overall validation MAE and RMSE.

All three ARIMA candidates perform slightly worse under the current
procedure: full-history training, no drift, fixed fitted parameters,
and daily state updates.

The performance differences are small. We have not established that
they are statistically distinguishable.

The naïve model remains the preferred candidate so far. Before final
selection, we will examine performance separately by validation year
and inspect forecast errors.

The final test period remains unused.

## 28. Check performance across validation years

Calculate MAE, RMSE, and mean error separately for 2023 and 2024.

Error = actual close - predicted close

Therefore:
- Positive mean error indicates average underprediction.
- Negative mean error indicates average overprediction.
- Mean error near zero does not imply small individual errors.

Compare each model with the naïve forecast within the same year.
Absolute errors across years can also reflect different price levels
and market volatility.

In [0]:
validation_predictions = {
    "Naïve: previous close": baseline_results,
    **candidate_predictions
}

yearly_rows = []

for model_name, results in validation_predictions.items():
    for year, group in results.groupby(results.index.year):
        errors = group["actual"] - group["prediction"]

        yearly_rows.append({
            "model": model_name,
            "year": int(year),
            "forecasts": len(group),
            "MAE": errors.abs().mean(),
            "RMSE": np.sqrt(errors.pow(2).mean()),
            "mean_error": errors.mean()
        })

yearly_comparison = pd.DataFrame(yearly_rows)

yearly_baseline = (
    yearly_comparison[
        yearly_comparison["model"] == "Naïve: previous close"
    ][["year", "MAE", "RMSE"]]
    .rename(columns={
        "MAE": "baseline_MAE",
        "RMSE": "baseline_RMSE"
    })
)

yearly_comparison = yearly_comparison.merge(
    yearly_baseline,
    on="year",
    validate="many_to_one"
)

yearly_comparison["MAE_improvement_pct"] = (
    100 * (
        1 - yearly_comparison["MAE"]
        / yearly_comparison["baseline_MAE"]
    )
)

yearly_comparison["RMSE_improvement_pct"] = (
    100 * (
        1 - yearly_comparison["RMSE"]
        / yearly_comparison["baseline_RMSE"]
    )
)

display(
    yearly_comparison[
        [
            "year", "model", "forecasts",
            "MAE", "RMSE", "mean_error",
            "MAE_improvement_pct", "RMSE_improvement_pct"
        ]
    ]
    .sort_values(["year", "MAE"])
    .reset_index(drop=True)
)

## 29. Inspect the naïve forecast's validation errors

Plot the errors of the current preferred candidate over time.

Look for:
- Persistent positive or negative errors.
- Clusters of large errors.
- Individual extreme misses.
- Changes in error magnitude.

These are out-of-sample, one-step-ahead validation errors, not
in-sample fitted residuals. The time plot alone cannot establish
whether the errors are uncorrelated.

In [0]:
fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(
    baseline_results.index,
    baseline_results["error"],
    color="steelblue",
    linewidth=0.8
)

ax.axhline(0, color="black", linewidth=1)

ax.set_title("Naïve forecast errors — validation period, 2023–2024")
ax.set_xlabel("Target date")
ax.set_ylabel("Actual − predicted close")
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

## 30. Initial observations from validation errors

The naïve forecast errors show larger absolute movements during parts
of 2024 than during much of 2023, with several extreme positive and
negative errors.

For this model, each forecast error equals the change between successive
closing prices.

The plot alone does not establish average bias or independence.
Differences in price level may also affect the size of price-unit errors.

We will examine error autocorrelation and use a Ljung–Box test as a
diagnostic check.

## 31. Check dependence in validation forecast errors

The error ACF measures linear correlation between errors at different
lags.

The Ljung–Box test jointly checks whether autocorrelations through a
specified lag are zero.

- Null hypothesis: autocorrelations through the selected lag are zero.
- A small p-value provides evidence against that hypothesis.
- A large p-value does not prove independence or forecast optimality.

We will inspect lags 10 and 20. These are related diagnostic checks,
not independent confirmations.

We will also plot the ACF of squared, mean-centered errors to investigate
dependence in error magnitude.

Changing volatility and extreme observations can affect these diagnostics.
Standard confidence bands and p-values should therefore be interpreted
cautiously.

In [0]:
from statsmodels.stats.diagnostic import acorr_ljungbox

errors = baseline_results["error"].dropna()
centered_errors = errors - errors.mean()

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

plot_acf(
    errors,
    lags=30,
    zero=False,
    alpha=0.05,
    ax=axes[0]
)
axes[0].set_title("ACF of naïve validation errors")
axes[0].set_ylim(-0.3, 0.3)

plot_acf(
    centered_errors.pow(2),
    lags=30,
    zero=False,
    alpha=0.05,
    ax=axes[1]
)
axes[1].set_title(
    "ACF of squared, mean-centered naïve validation errors"
)
axes[1].set_ylim(-0.3, 0.3)

for ax in axes:
    ax.set_xlabel("Lag (trading observations)")
    ax.set_ylabel("Correlation")
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

# The naïve benchmark has no fitted AR or MA coefficients.
ljung_box_results = acorr_ljungbox(
    errors,
    lags=[10, 20],
    model_df=0,
    return_df=True
)

ljung_box_results.index.name = "lags_tested"

display(ljung_box_results.reset_index())

print(f"Mean validation error: {errors.mean():.4f}")

## 32. Interpret the validation-error diagnostics

The Ljung–Box p-values are approximately:
- 0.4212 through lag 10.
- 0.6836 through lag 20.

At the 5% significance level, neither test rejects the hypothesis that
autocorrelations through the specified lag are zero.

One error-ACF spike near lag 10 crosses its pointwise confidence band.
An isolated crossing does not establish a useful forecasting pattern.

The squared-error ACF shows no obvious crossings of its displayed bands.
This does not prove independence or constant volatility.

These diagnostics concern 2023–2024 price-unit forecast errors. They
are not directly comparable with the earlier full-training-history
squared-log-return diagnostics.

The naïve forecast remains the preferred candidate so far. Year-specific
performance and average forecast bias still need to be reviewed.

In [0]:
display(
    yearly_comparison[
        [
            "year", "model", "forecasts",
            "MAE", "RMSE", "mean_error",
            "MAE_improvement_pct", "RMSE_improvement_pct"
        ]
    ]
    .sort_values(["year", "MAE"])
    .reset_index(drop=True)
)

print(
    "Overall naïve mean error:",
    round(baseline_results["error"].mean(), 6)
)

## 33. Conclusions from year-specific validation

The naïve forecast has the lowest MAE and RMSE in both 2023 and 2024
among the tested candidates.

Its positive mean error indicates average underprediction in both
years. This does not establish a persistent future bias.

We will not correct historical validation forecasts using the mean
error calculated from that same validation period.

The naïve forecast remains our preferred point-forecast candidate.
Next, we will evaluate a method for estimating prediction intervals.

## 34. Construct historical-quantile prediction intervals

For each validation target date:

1. Use the preceding 252 observed log returns.
2. Calculate their 5th and 95th percentiles.
3. Convert these return limits into prices using the previous close.

Lower limit = previous close × exp(5th-percentile log return)
Upper limit = previous close × exp(95th-percentile log return)

The target day's return is excluded from its own interval.
Previously observed validation returns may inform subsequent intervals.

These are candidate 90% prediction intervals, not confidence intervals
for a mean. Actual coverage may differ from 90%, particularly when
market conditions change.

We will assess:
- Coverage: percentage of actual closes inside the interval.
- Average interval width, in price units.
- Misses below and above the interval.

The final test period remains unused.

In [0]:
# Combine training and validation observations only.
interval_history = pd.concat([
    train_pd[["close"]],
    validation_pd[["close"]]
]).sort_index()

assert interval_history.index.is_unique
assert np.isfinite(interval_history["close"]).all()
assert (interval_history["close"] > 0).all()

historical_returns = np.log(interval_history["close"]).diff()

window = 252

# shift(1) excludes the target day's return from its interval.
lower_return = (
    historical_returns
    .rolling(window, min_periods=window)
    .quantile(0.05)
    .shift(1)
)

upper_return = (
    historical_returns
    .rolling(window, min_periods=window)
    .quantile(0.95)
    .shift(1)
)

previous_close = interval_history["close"].shift(1)

interval_results = pd.DataFrame({
    "actual": interval_history["close"],
    "prediction": previous_close,
    "lower": previous_close * np.exp(lower_return),
    "upper": previous_close * np.exp(upper_return)
}).loc[validation_pd.index].copy()

assert np.isfinite(interval_results.to_numpy()).all()
assert (interval_results["lower"] <= interval_results["upper"]).all()

interval_results["covered"] = (
    (interval_results["actual"] >= interval_results["lower"])
    & (interval_results["actual"] <= interval_results["upper"])
)

interval_results["width"] = (
    interval_results["upper"] - interval_results["lower"]
)

def summarize_intervals(data, period):
    return {
        "period": period,
        "forecasts": len(data),
        "coverage_pct": 100 * data["covered"].mean(),
        "average_width": data["width"].mean(),
        "below_lower_pct": (
            100 * (data["actual"] < data["lower"]).mean()
        ),
        "above_upper_pct": (
            100 * (data["actual"] > data["upper"]).mean()
        )
    }

interval_summary = [
    summarize_intervals(interval_results, "Overall")
]

for year, group in interval_results.groupby(interval_results.index.year):
    interval_summary.append(
        summarize_intervals(group, str(year))
    )

display(pd.DataFrame(interval_summary))

## 35. Interpret validation interval coverage

The candidate 90% intervals achieved:
- Overall coverage: 90.64%.
- 2023 coverage: 96.00%.
- 2024 coverage: 85.32%.

Overall coverage conceals substantial variation between years.
The intervals were relatively conservative in 2023 and undercovered
in 2024, despite becoming wider.

Lower- and upper-tail misses were reasonably balanced.

This method remains an initial uncertainty benchmark. Its validation
results do not justify claiming reliable 90% coverage in every period.

## 36. Freeze the procedure for final testing

The final evaluation will use observations dated 1 January 2025 onward.

Before viewing test performance, we fix:
- Point forecast: previous recorded trading day's close.
- Interval history: previous 252 observed log returns.
- Interval limits: empirical 5th and 95th percentiles.
- Forecast horizon: one trading observation ahead.
- Update schedule: update available history after each actual is observed.

Earlier test outcomes may inform later forecasts, as they would in
daily use. Each target observation must remain excluded from its own
forecast and interval.

We will report point accuracy and interval coverage without changing
the procedure in response to these test results.

Coverage for partial calendar years will be identified by their actual
date range.

In [0]:
test_pd = (
    test_prices
    .select("date", "close")
    .orderBy("date")
    .toPandas()
)

test_pd["date"] = pd.to_datetime(test_pd["date"])
test_pd = test_pd.set_index("date")

# Combine available history, preserving chronological order.
final_history = pd.concat([
    train_pd[["close"]],
    validation_pd[["close"]],
    test_pd[["close"]]
]).sort_index()

assert final_history.index.is_unique
assert validation_pd.index.max() < test_pd.index.min()
assert np.isfinite(final_history["close"]).all()
assert (final_history["close"] > 0).all()

log_returns = np.log(final_history["close"]).diff()
previous_close = final_history["close"].shift(1)

# The shift excludes each target day's actual return.
lower_return = (
    log_returns.rolling(252, min_periods=252)
    .quantile(0.05)
    .shift(1)
)

upper_return = (
    log_returns.rolling(252, min_periods=252)
    .quantile(0.95)
    .shift(1)
)

final_test_results = pd.DataFrame({
    "actual": final_history["close"],
    "prediction": previous_close,
    "lower": previous_close * np.exp(lower_return),
    "upper": previous_close * np.exp(upper_return)
}).loc[test_pd.index].copy()

assert np.isfinite(final_test_results.to_numpy()).all()
assert (
    final_test_results["lower"] <= final_test_results["upper"]
).all()

final_test_results["error"] = (
    final_test_results["actual"] - final_test_results["prediction"]
)

final_test_results["covered"] = (
    (final_test_results["actual"] >= final_test_results["lower"])
    & (final_test_results["actual"] <= final_test_results["upper"])
)

final_test_results["width"] = (
    final_test_results["upper"] - final_test_results["lower"]
)

def summarize_final_test(data, period):
    return {
        "period": period,
        "first_date": data.index.min().date().isoformat(),
        "last_date": data.index.max().date().isoformat(),
        "forecasts": len(data),
        "MAE": data["error"].abs().mean(),
        "RMSE": np.sqrt(data["error"].pow(2).mean()),
        "mean_error": data["error"].mean(),
        "coverage_pct": 100 * data["covered"].mean(),
        "average_width": data["width"].mean(),
        "below_lower_pct": (
            100 * (data["actual"] < data["lower"]).mean()
        ),
        "above_upper_pct": (
            100 * (data["actual"] > data["upper"]).mean()
        )
    }

test_summary = [
    summarize_final_test(final_test_results, "Overall test")
]

for year, group in final_test_results.groupby(
    final_test_results.index.year
):
    test_summary.append(
        summarize_final_test(group, str(year))
    )

display(pd.DataFrame(test_summary))

## 37. Final test findings

The frozen forecasting procedure was evaluated on 414 observations,
from 2 January 2025 through 27 August 2026.

Point-forecast results:
- MAE: 4.4037 price units.
- RMSE: 7.2174 price units.
- Mean error: 0.0458 price units.

Prediction-interval results:
- Nominal coverage: 90%.
- Observed coverage: 83.33%.
- Average width: 13.0875 price units.

Coverage declined from 87.20% in 2025 to 77.44% in the partial 2026
period, despite wider intervals.

The historical-quantile interval method did not maintain its nominal
coverage on the held-out test period.

The naïve point forecast was selected using validation performance.
These test results do not establish that it outperformed untested
alternatives during the test period.

Conclusions remain conditional on the supplied dataset's accuracy
and its unconfirmed price-adjustment conventions.

## 38. Audit the largest final-test errors

Identify the ten largest absolute forecast errors and their contribution
to total squared error.

Large errors may reflect genuine market movements, corporate actions,
or source-data problems. They will remain in the reported results unless
independent evidence establishes an error.

This inspection occurs after final testing. Any model revisions informed
by these outcomes will require fresh evaluation data.

In [0]:
test_error_audit = final_test_results.copy()

test_error_audit["absolute_error"] = (
    test_error_audit["error"].abs()
)

test_error_audit["squared_error"] = (
    test_error_audit["error"].pow(2)
)

# For the naïve forecast, prediction is the previous close.
test_error_audit["price_change_pct"] = (
    100 * (
        test_error_audit["actual"]
        / test_error_audit["prediction"] - 1
    )
)

largest_errors = test_error_audit.nlargest(
    10, "absolute_error"
)

display(
    largest_errors[
        [
            "actual", "prediction", "error",
            "price_change_pct", "lower", "upper", "covered"
        ]
    ].reset_index()
)

total_squared_error = test_error_audit["squared_error"].sum()

top_10_share = (
    100 * largest_errors["squared_error"].sum()
    / total_squared_error
    if total_squared_error > 0
    else 0.0
)

print(
    f"Top 10 errors account for {top_10_share:.2f}% "
    "of total test squared error."
)

## 39. Findings from the extreme-error audit

The ten largest errors account for approximately:
- 54.49% of total test squared error.
- 16.96% of total test absolute error.

The observation on 14 July 2026 alone accounts for approximately
24.82% of total squared error.

This concentration explains the sensitivity of RMSE to extreme moves.
The original scores remain unchanged.

All ten observations were outside their prediction intervals.
However, there were 69 interval misses overall, leaving 59 misses
outside the top-ten error list. Undercoverage therefore extends beyond
the most extreme observations.

Large observations will not be removed without evidence of data errors.

## 40. Experiment conclusion and limitations

The naïve previous-close forecast was selected over three ARIMA
candidates based on validation MAE and RMSE.

On the held-out test period, it achieved:
- MAE: 4.4037 price units.
- RMSE: 7.2174 price units.

The candidate 90% historical-quantile intervals achieved only 83.33%
coverage, with particularly low coverage during the partial 2026 period.
They should not be presented as reliably calibrated 90% intervals.

Limitations include:
- Unconfirmed source price-adjustment conventions.
- A single stock and one validation/test arrangement.
- Fixed-parameter, full-history ARIMA candidates without drift.
- Sensitivity to extreme movements and changing return distributions.

The test period has now been examined. Any changes motivated by these
results are exploratory and require fresh evaluation data for an
independent assessment.